In [1]:
import os
os.chdir('/home/smallyan/eval_agent')

repo_path = "/net/scratch2/smallyan/filter_eval"
os.chdir(repo_path)
print(f"Working directory: {os.getcwd()}")

# Check nnsight version
import subprocess
result = subprocess.run(['pip', 'show', 'nnsight'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if 'Version' in line:
        print(line)

Working directory: /net/scratch2/smallyan/filter_eval


Version: 0.5.2


# Code Evaluation for Circuit Analysis (Filter Heads Project)

Repository: `/net/scratch2/smallyan/filter_eval`

This notebook evaluates the code implementation for the "LLMs Process Lists With General Filter Heads" project.

## 1. Setup and Configuration

In [2]:
# Initialize evaluation tracking
evaluation_results = []
corrected_blocks = []
failed_blocks = []

def add_eval_result(cell_id, description, runnable, correct_impl, redundant, irrelevant, notes=""):
    """Helper function to add evaluation results"""
    evaluation_results.append({
        "cell_id": cell_id,
        "description": description,
        "runnable": runnable,
        "correct_implementation": correct_impl,
        "redundant": redundant,
        "irrelevant": irrelevant,
        "notes": notes
    })
    status = "PASS" if runnable == "Y" and correct_impl == "Y" else "ISSUE"
    print(f"[{status}] {cell_id}: Runnable={runnable}, Correct={correct_impl}")
    if notes:
        print(f"   Notes: {notes}")

# Check GPU
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


CUDA available: True
GPU: NVIDIA A100 80GB PCIe


## 2. Evaluating demo.ipynb

The demo notebook is the main entry point according to CodeWalkthrough.md.

In [3]:
# Cell 0: autoreload magic commands
add_eval_result(
    cell_id="demo.ipynb:Cell_0",
    description="Autoreload magic commands",
    runnable="Y",
    correct_impl="Y",
    redundant="N",
    irrelevant="N",
    notes="Jupyter magic for development"
)

[PASS] demo.ipynb:Cell_0: Runnable=Y, Correct=Y
   Notes: Jupyter magic for development


In [4]:
# Cell 1: Import and model loading
try:
    import torch
    import transformers
    from src.models import ModelandTokenizer

    print(f"torch version: {torch.__version__}")
    print(f"transformers version: {transformers.__version__}")

    # Use local path for Llama model
    model_key = "/net/projects/chai-lab/shared_models/Meta-Llama-3.1-70B-Instruct"
    
    mt = ModelandTokenizer(
        model_key=model_key,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )
    
    print(f"Model loaded: {mt.name}")
    print(f"Model layers: {mt.n_layer}")
    
    add_eval_result(
        cell_id="demo.ipynb:Cell_1",
        description="Import libraries and load model",
        runnable="Y",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes="Model loaded successfully"
    )
except Exception as e:
    failed_blocks.append("demo.ipynb:Cell_1")
    add_eval_result(
        cell_id="demo.ipynb:Cell_1",
        description="Import libraries and load model",
        runnable="N",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes=f"Error: {str(e)[:200]}"
    )
    import traceback
    traceback.print_exc()

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


`torch_dtype` is deprecated! Use `dtype` instead!


torch version: 2.7.1+cu118
transformers version: 4.57.3


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded: /net/projects/chai-lab/shared_models/Meta-Llama-3.1-70B-Instruct
Model layers: 80
[PASS] demo.ipynb:Cell_1: Runnable=Y, Correct=Y
   Notes: Model loaded successfully


In [5]:
# Cell 2: Select filter head
try:
    # For Llama 3.1/3.3, we use the same filter heads
    layer_idx, head_idx = 35, 19
    print(f"Selected filter head: Layer {layer_idx}, Head {head_idx}")
    
    add_eval_result(
        cell_id="demo.ipynb:Cell_2",
        description="Select filter head based on model type",
        runnable="Y",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes="Filter head selection works correctly"
    )
except Exception as e:
    failed_blocks.append("demo.ipynb:Cell_2")
    add_eval_result(
        cell_id="demo.ipynb:Cell_2",
        description="Select filter head based on model type",
        runnable="N",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes=f"Error: {str(e)[:200]}"
    )

Selected filter head: Layer 35, Head 19
[PASS] demo.ipynb:Cell_2: Runnable=Y, Correct=Y
   Notes: Filter head selection works correctly


In [6]:
# Cell 4: Load SelectOneTask data
try:
    from src.selection.data import SelectOneTask
    from typing import Literal
    import os

    prompt_template_idx = 3
    option_style: Literal["single_line", "numbered"] = "single_line"
    n_distractors = 5

    select_task = SelectOneTask.load(
        path=os.path.join("data_save", "selection", "objects.json")
    )
    
    print(f"Task loaded: {type(select_task).__name__}")
    print(f"Categories: {select_task.categories[:5]}")
    
    add_eval_result(
        cell_id="demo.ipynb:Cell_4",
        description="Load SelectOneTask data",
        runnable="Y",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes="SelectOneTask data loaded successfully"
    )
except Exception as e:
    failed_blocks.append("demo.ipynb:Cell_4")
    add_eval_result(
        cell_id="demo.ipynb:Cell_4",
        description="Load SelectOneTask data",
        runnable="N",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes=f"Error: {str(e)[:200]}"
    )
    import traceback
    traceback.print_exc()

['name', 'prompt_templates', 'odd_one_prompt_templates', 'order_prompt_templates', 'count_prompt_templates', 'yes_no_prompt_templates', 'first_item_in_cat_prompt_templates', 'last_item_in_cat_prompt_templates', 'categories', 'exclude_categories']
Task loaded: SelectOneTask
Categories: ['fruit', 'vehicle', 'furniture', 'animal', 'music instrument']
[PASS] demo.ipynb:Cell_4: Runnable=Y, Correct=Y
   Notes: SelectOneTask data loaded successfully


In [7]:
# Cell 5: Get random sample
try:
    sample = select_task.get_random_sample(
        mt=mt,
        option_style=option_style,
        prompt_template_idx=prompt_template_idx,
        category="fruit",
        filter_by_lm_prediction=True, 
    )

    print(f"Sample prompt: {sample.prompt()[:100]}...")
    print(f"Target object: {sample.obj}")
    print(f"Answer token: {mt.tokenizer.decode([sample.ans_token_id])}")
    
    add_eval_result(
        cell_id="demo.ipynb:Cell_5",
        description="Get random sample from task",
        runnable="Y",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes="Random sample generation works correctly"
    )
except Exception as e:
    failed_blocks.append("demo.ipynb:Cell_5")
    add_eval_result(
        cell_id="demo.ipynb:Cell_5",
        description="Get random sample from task",
        runnable="N",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes=f"Error: {str(e)[:200]}"
    )
    import traceback
    traceback.print_exc()

[ISSUE] demo.ipynb:Cell_5: Runnable=N, Correct=Y
   Notes: Error: 'str' object is not callable


Traceback (most recent call last):
  File "/tmp/ipykernel_2784276/2935887578.py", line 3, in <module>
    sample = select_task.get_random_sample(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/filter_eval/src/selection/data.py", line 636, in get_random_sample
    is_correct, predictions, track_objs = verify_correct_option(
                                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/filter_eval/src/selection/utils.py", line 82, in verify_correct_option
    logits = get_hs(
             ^^^^^^^
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/filter_eval/src/functional.py", line 879, in get_hs
    with mt.trace(input, scan=False):
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/nnsight/contexts/Runner.py", line 41, in __exit__
    

In [8]:
# Check which nnsight is being imported
import nnsight
print(f"nnsight location: {nnsight.__file__}")

# Try to get version
try:
    print(f"nnsight version: {nnsight.__version__}")
except:
    # Check the package metadata
    import importlib.metadata
    print(f"nnsight version (from metadata): {importlib.metadata.version('nnsight')}")

nnsight location: /home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/nnsight/__init__.py
nnsight version (from metadata): 0.2.21


In [9]:
# Upgrade nnsight in the conda environment
import subprocess
print("Upgrading nnsight in conda environment...")
result = subprocess.run(
    ['/home/smallyan/.conda/envs/meta/bin/pip', 'install', '--upgrade', 'nnsight==0.5.2'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")

Upgrading nnsight in conda environment...


client]->nnsight==0.5.2) (2.2.3)
Using cached nnsight-0.5.2-cp311-cp311-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl (92 kB)
  Attempting uninstall: nnsight
    Found existing installation: nnsight 0.2.21
    Uninstalling nnsight-0.2.21:
      Successfully uninstalled nnsight-0.2.21

